# Pilot 2025 Calibration, Estimability, and Natural-Must MBDoE

This notebook applies the reduced extended fermentation model to the pilot-scale natural-must dataset. It mirrors the laboratory workflow, but the design space is intentionally narrower: natural must is assumed, and the realistic manipulated inputs are temperature setpoints and nutrient additions.

The model combines primary fermentation, glycerol production, the reduced secondary v2 model, and empirical aroma synthesis for ethyl acetate, isoamyl acetate, and ethyl octanoate.

## Mathematical Structure

The primary state vector is

$$x_p = [X, X_d, N, G, F, E, Gly]^T$$

where `X` is viable biomass, `X_d` is dead biomass, `N` is assimilable nitrogen, `G` and `F` are glucose and fructose, `E` is ethanol, and `Gly` is glycerol.

The secondary model uses

$$x_s = [Pyr, AcAld, Acetate, O_2, CO_2]^T$$

with the reduced v2 free set

$$\theta_s = \{k_{PyrS,N}, k_{PyrO2}, k_{PyrDrain}, k_{AldS,N}, k_{AldRed}, k_{AcAld}, k_{AcStress}\}.$$

Aroma synthesis is empirical and rate-dependent:

$$r_i = \left(k_{i,growth}\phi_N + k_{i,stationary}(1-\phi_N)\right)q_S,$$

where `i` is ethyl acetate, isoamyl acetate, or ethyl octanoate; `q_S` is the total sugar uptake rate; and `\phi_N` is the nitrogen-growth phase proxy.

Volatilization is represented as an effective liquid-to-condenser transfer:

$$r_{loss,i} = \alpha_i K_i(T,E,S) q_{CO2} C_{L,i}.$$

The observation model uses

$$C_{wine,i}^{obs} = C_{total,i}^{obs} - C_{cond,i}^{obs}, \qquad C_{cond,i}^{obs} = C_{cond,i}.$$

Therefore the pilot condenser data make the `\alpha_i` loss parameters estimable as effective volatilization/capture coefficients.

In [ ]:
from pathlib import Path
import pandas as pd
cwd = Path.cwd()
if (cwd / 'results' / 'calibration_estimability').exists():
    RESULTS = cwd / 'results' / 'calibration_estimability'
else:
    RESULTS = cwd / 'fermentation_model' / 'pilot_2025' / 'results' / 'calibration_estimability'
fit_summary = pd.read_csv(RESULTS / 'fit_summary.csv')
theta = pd.read_csv(RESULTS / 'theta_pilot_extended.csv', index_col=0)
estimability = pd.read_csv(RESULTS / 'parameter_estimability_current.csv')
weak = pd.read_csv(RESULTS / 'weak_directions_current.csv')
ranking = pd.read_csv(RESULTS / 'candidate_ranking_natural.csv')
selected = pd.read_csv(RESULTS / 'selected_campaign_hybrid.csv')
fit_summary

## Calibrated Parameter Vector

In [ ]:
theta

## Current-Data Estimability

The FIM is computed from log-parameter finite differences of the full residual vector. The approximate standard deviation is therefore in log-parameter space. Large `approx_95_multiplier` values indicate broad practical uncertainty. Parameters on active bounds are not considered reliable even when the local curvature appears high.

In [ ]:
estimability.sort_values('std_log_approx', ascending=False)

## Weak Eigen-Directions

Weak directions identify combinations of parameters that the current pilot data do not separate well. These directions are more informative than single-parameter rankings when parameters are correlated.

In [ ]:
weak

## Natural-Must Candidate Design Ranking

Candidate experiments are restricted to natural must. The manipulated variables are temperature setpoint profiles and nitrogen pulse timing/dose. No glucose, fructose, ethanol, or biomass injections are included in this pilot-scale design library.

In [ ]:
cols = ['candidate', 'family', 'combined_logdet', 'combined_min_relative_eigenvalue', 'secondary_aroma_mean_var_reduction', 'secondary_aroma_worst_var_reduction', 'N_pulses_kg_m3', 'rationale']
ranking[cols].head(10)

## Selected Hybrid Campaign

The hybrid objective keeps D-optimality as the information-volume term and penalizes designs that leave very weak eigen-directions. This is the same practical logic used in the laboratory design fork.

In [ ]:
selected[['campaign_order', 'candidate', 'family', 'campaign_logdet', 'campaign_min_relative_eigenvalue', 'secondary_aroma_mean_var_reduction', 'secondary_aroma_worst_var_reduction', 'N_pulses_kg_m3', 'rationale']]

## Generated Plots

Calibration fit plots are saved in `results/calibration_estimability/plots/fits`. Selected design input plots are saved in `results/calibration_estimability/plots/designs`.